In [79]:
import os
import pandas as pd
import matplotlib as mp

### Best Weights

In [80]:
df = pd.read_csv("best_weights.csv")

df.head
len(df)

6980

In [81]:
queries_data = "/Users/hannahzhang/Desktop/Github Repos/ERSP-TeamYang/data/queries/queries.dev.tsv"

In [82]:
queries_df = pd.read_csv(queries_data, sep="\t", names=['Query Id', 'Query'])

print(queries_df)


        Query Id                            Query
0        1048578   cost of endless pools/swim spa
1        1048579                     what is pcnt
2        1048580                what is pcb waste
3        1048581                    what is pbis?
4        1048582                   what is paysky
...          ...                              ...
101088    480594  price of copper by ounce, pound
101089    524271  trazodone for dogs side effects
101090   1048565    who plays sebastian michaelis
101091   1048570     what is pearls before swine?
101092    524285        treadmill incline meaning

[101093 rows x 2 columns]


### Word/ Character Count

In [83]:
queries = df['Query ID']

char_count_list = []
word_count_list = []

for query in queries:
    query = queries_df.loc[queries_df["Query Id"] == query]["Query"].to_string(index=False).strip()

    char_count = 0
    for char in query:
        if char.isalnum():
            char_count += 1

    word_list = query.split(" ")
    word_count = len(word_list)

    char_count_list.append(char_count)
    word_count_list.append(word_count)


df['Character Count'] = char_count_list 
df['Word Count'] = word_count_list

df.head()

,Query ID,Best Alpha,MRR,Character Count,Word Count
0,2,0.00,1.00,22,3
1,1048585,0.00,0.50,23,5
2,458771,0.00,1.00,26,5
3,163860,0.01,0.25,22,4
4,458774,0.00,0.50,23,4


### Question Type Keywords (what, when, etc...)

In [90]:
what_list = [0] * 6980
where_list = [0] * 6980
when_list = [0] * 6980
why_list = [0] * 6980
how_list = [0] * 6980

queries_subset = queries_df[queries_df["Query Id"].isin(queries)]["Query"]

for idx in range(len(queries_subset)):
    word_list = queries_subset.iloc[idx].lower().split(" ")

    if "what" in word_list:
        what_list[idx] = 1

    if "where" in word_list:
        where_list[idx] = 1

    if "when" in word_list:
        when_list[idx] = 1

    if "why" in word_list:
        why_list[idx] = 1

    if "how" in word_list:
        how_list[idx] = 1

df['What'] = what_list
df['Where'] = where_list
df['When'] = when_list
df['Why'] = why_list
df['How'] = how_list

df


,Query ID,Best Alpha,MRR,Character Count,Word Count,What,Where,When,Why,How
0,2,0.00,1.00,22,3,1,0,0,0,0
1,1048585,0.00,0.50,23,5,0,0,0,0,0
2,458771,0.00,1.00,26,5,0,0,0,0,0
3,163860,0.01,0.25,22,4,1,0,0,0,0
4,458774,0.00,0.50,23,4,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
6975,884722,0.00,1.00,41,7,1,0,0,0,0
6976,393203,0.00,0.20,23,6,0,0,0,0,0
6977,196596,0.01,0.50,26,3,0,1,0,0,0
6978,1048565,0.01,0.20,26,4,1,0,0,0,0


In [91]:
questions = queries_df["Query"].to_list()
for question in questions:
    print(question)

cost of endless pools/swim spa
what is pcnt
what is pcb waste
what is pbis?
what is paysky
what is paydata
what is pay range for warehouse specialist in minneapolis
what is paula deen's brother
what is paul gum disease
what is patron
how long is a college hockey game
 Androgen receptor define
treasuries cost basis
treasury officer hong kong average salary
what is patricia cornwell's latest book
what is pastoral medicine
what is past perfect present tense
 definition of Executive Officer defined by FRB 
treating diabetes
what is president lincoln known for
what is passage door levers
how long is a cosmic giga second
treating powdery mildew on squash
treating tension headaches without medication
who plays storm in days of future past
how long is a creative writing major
what is pascal's law in simple terms
what is parkland near in florida
what is parkinson, dies
 pneumo is a prefix meaning air. Knowing this, explain why this condition is called pneumothorax
treatment and prognosis for lu

### Question Type

In [96]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

question_type = []

# Load model
model_name = "PrimeQA/tydiqa-boolean-question-classifier"
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name)

for idx in range(len(queries_subset)):
    question = queries_subset.iloc[idx]

    # Tokenize input question
    inputs = tokenizer(question, return_tensors="pt", truncation=True, padding=True, max_length=512)

    # Get model's output
    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits

    # Convert logits to probabilities using softmax
    probabilities = torch.nn.functional.softmax(logits, dim=-1)

    # Get the predicted class
    predicted_class = torch.argmax(probabilities, dim=-1).item()
    question_type.append(predicted_class)

In [101]:
df["Question Type"] = question_type
df

,Query ID,Best Alpha,MRR,Character Count,Word Count,What,Where,When,Why,How,Question Type
0,2,0.00,1.00,22,3,1,0,0,0,0,0
1,1048585,0.00,0.50,23,5,0,0,0,0,0,1
2,458771,0.00,1.00,26,5,0,0,0,0,0,1
3,163860,0.01,0.25,22,4,1,0,0,0,0,0
4,458774,0.00,0.50,23,4,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
6975,884722,0.00,1.00,41,7,1,0,0,0,0,0
6976,393203,0.00,0.20,23,6,0,0,0,0,0,0
6977,196596,0.01,0.50,26,3,0,1,0,0,0,0
6978,1048565,0.01,0.20,26,4,1,0,0,0,0,0


### KNN

In [22]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_squared_error, r2_score

# Separate data into features and target
X = df.drop(['Best Alpha', 'Query ID'], axis=1)
y = df['Best Alpha']

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Scale features using StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Initialize model
knn = KNeighborsRegressor(n_neighbors=3)

# Fit the model
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f'Mean Squared Error: {mse}')
print(f'R-squared: {r2}')


Mean Squared Error: 0.0011154489016236865
R-squared: -0.3067147529610308
